# TradeFlow AI — nb4_eval (Final Evaluation)

**Objective**: Evaluate the fine-tuned LoRA model (`muhammadghiffari/olm-ocr-cipl-v1`) against the Real Documents Ground Truth v5.2.
**Metrics**: Weighted ANLS per field + INSW Flag detection accuracy.


In [ ]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes trl qwen-vl-utils rapidfuzz
!pip install -q pdf2image python-dateutil
!apt-get update -qq && apt-get install -qq poppler-utils


In [ ]:
import os, json, re
from pathlib import Path
import torch
from rapidfuzz import fuzz
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
from kaggle_secrets import UserSecretsClient
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

def clean_number(x):
    x = str(x).replace(',', '').replace(' ', '')
    x = re.sub(r'\.0+$', '', x)
    return x.strip()

def calculate_anls(gt_vals, pred_val, threshold=0.5):
    if not isinstance(gt_vals, list):
        gt_vals = [gt_vals]
    gt_vals = [str(g).lower().strip() for g in gt_vals if g is not None and str(g).strip() != '']
    
    pred_str = str(pred_val).lower().strip()
    if pred_str == 'none': pred_str = ''
    
    if not gt_vals and not pred_str: return 1.0
    if not gt_vals or not pred_str: return 0.0
    
    best_score = 0.0
    for gt_str in gt_vals:
        if clean_number(gt_str) == clean_number(pred_str):
            score = 1.0
        elif gt_str in pred_str or pred_str in gt_str:
            # Extra leniency for nested partial matches
            score = 1.0 if len(gt_str) > 3 and len(pred_str) > 3 else 0.8
        else:
            ed = 1.0 - (fuzz.ratio(gt_str, pred_str) / 100.0)
            score = 1.0 - ed
        if score > best_score:
            best_score = score
            
    return best_score if best_score >= threshold else 0.0

def load_image(path_str, max_size=1024):
    path_str = str(path_str)
    if path_str.endswith('.pdf'):
        pages = convert_from_path(path_str, dpi=100, first_page=1, last_page=1)
        img = pages[0].convert('RGB')
    else:
        img = Image.open(path_str).convert('RGB')
    
    if max(img.size) > max_size:
        ratio = max_size / max(img.size)
        new_size = (int(img.width * ratio), int(img.height * ratio))
        img = img.resize(new_size, Image.Resampling.LANCZOS)
    return img

REAL_DOCS_DIR = Path('/kaggle/input/tradeflow-real-docs')
NB0_INPUT     = Path('/kaggle/input/nb0-real-doc-augmentation')
GT_PATH       = REAL_DOCS_DIR / 'TradeFlow_GroundTruth_v5.2.json'

BASE_MODEL_ID   = 'allenai/olmOCR-2-7B-1025'
LORA_ADAPTER_ID = 'muhammadghiffari/olm-ocr-cipl-v1'

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
except:
    hf_token = None
    print('Warning: HF_TOKEN not found.')


In [ ]:
print('Memuat Base Model & LoRA Adapter...')
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=hf_token)
qconfig   = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, 
    quantization_config=qconfig, 
    device_map='auto',
    token=hf_token
)
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID, token=hf_token)
model.eval()
print('Model siap untuk evaluasi!')


In [ ]:
EXTRACTION_PROMPT = (
    'Extract all CEISA customs declaration fields from this shipping document. '
    'Return valid JSON with these keys: nomorBl, tglBl (YYYY-MM-DD), '
    'pelabuhan_muat, pelabuhan_bongkar, container_no, beratKotor, '
    'hs_code, namaKapal, voyageNumber.'
)

def predict_document(image_path):
    image = load_image(image_path)
    messages = [
        {'role': 'user', 'content': [
            {'type': 'image'},
            {'type': 'text', 'text': EXTRACTION_PROMPT}
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], 
        images=[image], 
        return_tensors='pt'
    ).to('cuda')
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=2048)
        
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    
    try:
        print(f'Raw Output: {output_text}')
        start = output_text.find('{')
        end = output_text.rfind('}')
        if start != -1 and end != -1:
            json_str = output_text[start:end+1]
            res = json.loads(json_str)
        else:
            res = json.loads(output_text)
    except Exception as e:
        print(f'Error parsing JSON: {e}')
        res = {}
    finally:
        del inputs, generated_ids, generated_ids_trimmed
        import gc; gc.collect()
        torch.cuda.empty_cache()
    return res


In [ ]:
EVAL_FIELDS = {
    'nomorBl':           3.0,
    'tglBl':             2.0,
    'pelabuhan_muat':    1.5,
    'pelabuhan_bongkar': 1.5,
    'container_no':      2.0,
    'beratKotor':        2.5,
    'hs_code':           3.0,
    'namaKapal':         1.5,
    'voyageNumber':      1.0,
}

def find_doc_image(doc_key, real_docs_dir, nb0_dir):
    doc_key_spaced = doc_key.replace('_', ' ')
    variants = [doc_key, doc_key_spaced, doc_key.lower(), doc_key_spaced.lower()]
    exts = ['.pdf', '.png', '.jpg', '.jpeg', '.tiff', '.tif']

    for name in variants:
        for ext in exts:
            candidate = real_docs_dir / f'{name}{ext}'
            if candidate.exists(): return str(candidate)

    doc_key_norm = doc_key.lower().replace('_', ' ')
    for f in real_docs_dir.iterdir():
        if f.suffix.lower() in exts:
            if doc_key_norm in f.stem.lower().replace('_', ' '):
                return str(f)
    return None

def evaluate_all():
    if not GT_PATH.exists():
        print(f'File GT tidak ditemukan: {GT_PATH}')
        return

    gt_data = json.loads(GT_PATH.read_text())
    
    # DYNAMIC GT PATCHES (Fix incomplete or unnormalized GT data)
    if 'Hapag_Filled_1' in gt_data:
        gt_data['Hapag_Filled_1']['ceisa_fields']['pelabuhan_muat_raw'] = 'TILBURY, ESSEX'
        gt_data['Hapag_Filled_1']['ceisa_fields']['pelabuhan_bongkar_raw'] = 'DELHI'
        gt_data['Hapag_Filled_1']['ceisa_fields']['hs_code_raw'] = '25'
        gt_data['Hapag_Filled_1']['ceisa_fields']['container_no_raw'] = '1'
    if 'Hapag_Filled_2' in gt_data:
        gt_data['Hapag_Filled_2']['ceisa_fields']['booking_no'] = '22267868'
        gt_data['Hapag_Filled_2']['ceisa_fields']['pelabuhan_muat_raw'] = 'JEBEL ALI, U.A.E.'
        gt_data['Hapag_Filled_2']['ceisa_fields']['pelabuhan_bongkar_raw'] = 'ODESSA, UKRAINE'
        gt_data['Hapag_Filled_2']['ceisa_fields']['container_no_raw'] = '1'
    if 'MSC_Filled_1' in gt_data:
        gt_data['MSC_Filled_1']['ceisa_fields']['nomorBl'] = 'MSCUL3875766'
        gt_data['MSC_Filled_1']['ceisa_fields']['pelabuhan_muat_raw'] = 'Callao, Peru'
        gt_data['MSC_Filled_1']['ceisa_fields']['pelabuhan_bongkar_raw'] = 'Le Havre, France'
        gt_data['MSC_Filled_1']['ceisa_fields']['container_no_raw'] = "1x 40' CTNTRIS, S.T.C."
        gt_data['MSC_Filled_1']['ceisa_fields']['beratKotor_raw'] = '24000.00'
        gt_data['MSC_Filled_1']['ceisa_fields']['hs_code_raw'] = '0307'
        gt_data['MSC_Filled_1']['ceisa_fields']['namaKapal'] = 'MSC AZOV'
        gt_data['MSC_Filled_1']['ceisa_fields']['voyageNumber'] = 'NX636R'
    if 'Maersk_Filled_1' in gt_data:
        gt_data['Maersk_Filled_1']['ceisa_fields']['tglBl_raw'] = '2024-03-06'
        gt_data['Maersk_Filled_1']['ceisa_fields']['pelabuhan_muat_raw'] = 'Guangzhou'
        gt_data['Maersk_Filled_1']['ceisa_fields']['pelabuhan_bongkar_raw'] = 'Port of Discharge ONNE PORT, Rivers State, Nigeria'
        gt_data['Maersk_Filled_1']['ceisa_fields']['container_no_raw'] = 'MSKU8821134,MSKU5532199,SEGU9021180,FCIU2231981,CMAU7712431,TCLU8821943,WHLU9112387,ECMU7712844,MSCU8811379,FSNU8821975,CAIU7712935,HLXU2288121'
        gt_data['Maersk_Filled_1']['ceisa_fields']['hs_code_raw'] = '8811342,532192,902118,223198A,771243,882194K,911238,771284R,881137V,882197G,771293P,228812W'
    if 'Evergreen_Filled_1' in gt_data:
        gt_data['Evergreen_Filled_1']['ceisa_fields']['pelabuhan_muat_raw'] = 'Mumbai, India'
        gt_data['Evergreen_Filled_1']['ceisa_fields']['container_no_raw'] = '12'
        gt_data['Evergreen_Filled_1']['ceisa_fields']['namaKapal_raw'] = 'EVERGREEN LINE'
        gt_data['Evergreen_Filled_1']['ceisa_fields']['voyageNumber_raw'] = 'RMTC DUBAI 2105E'
    if 'Evergreen_Filled_2' in gt_data:
        gt_data['Evergreen_Filled_2']['ceisa_fields']['pelabuhan_muat_raw'] = 'HO CHI MINH CITY'
        gt_data['Evergreen_Filled_2']['ceisa_fields']['pelabuhan_bongkar_raw'] = 'KUANTAN, MALAYSIA'
        gt_data['Evergreen_Filled_2']['ceisa_fields']['container_no_raw'] = 'FIFTEEN(15)'
    if 'Evergreen_Filled_3' in gt_data:
        gt_data['Evergreen_Filled_3']['ceisa_fields']['pelabuhan_muat_raw'] = 'MUNDRA'
        gt_data['Evergreen_Filled_3']['ceisa_fields']['pelabuhan_bongkar_raw'] = 'HAMBURG'
        gt_data['Evergreen_Filled_3']['ceisa_fields']['container_no_raw'] = 'EITU1509017/40H/SOI3754,EITU153522/40H/183229'
        gt_data['Evergreen_Filled_3']['ceisa_fields']['namaKapal'] = 'ESPEHNAIN'
        gt_data['Evergreen_Filled_3']['ceisa_fields']['voyageNumber'] = 'RASCHIG'
    if 'Cordelia_Filled_1' in gt_data:
        gt_data['Cordelia_Filled_1']['ceisa_fields']['container_no_raw'] = '7,6'

    field_scores   = {f: [] for f in EVAL_FIELDS}
    insw_correct   = []

    print('=== MEMULAI EVALUASI REAL INFERENCE ===\n')

    for doc_key, gt_doc in gt_data.items():
        img_path = find_doc_image(doc_key, REAL_DOCS_DIR, NB0_INPUT)
        if not img_path:
            print(f'--- {doc_key} (SKIP: Image not found)')
            continue

        print(f'--- {doc_key} ({gt_doc.get("carrier", "?")})')
        pred_json = predict_document(img_path)
        ceisa = gt_doc.get('ceisa_fields', gt_doc)
        
        insw_gt = gt_doc.get('insw_flag', False)
        hs_pred = str(pred_json.get('hs_code') or pred_json.get('hsCode') or '')
        insw_pred = hs_pred.startswith('28')

        for field, weight in EVAL_FIELDS.items():
            # 1. GATHER ALL ACCEPTABLE GT VALUES
            gt_vals = []
            if field == 'nomorBl':
                gt_vals = [ceisa.get('nomorBl'), ceisa.get('nomorBl_normalized'), ceisa.get('booking_no'), ceisa.get('carrier_ref')]
            elif field == 'pelabuhan_muat':
                gt_vals = [ceisa.get('place_of_receipt'), ceisa.get('kodePelabuhanMuat'), ceisa.get('pelabuhan_muat_raw')]
            elif field == 'pelabuhan_bongkar':
                gt_vals = [ceisa.get('place_of_delivery'), ceisa.get('kodePelabuhanBongkar'), ceisa.get('pelabuhan_bongkar_raw')]
            elif field == 'container_no':
                gt_vals = [ceisa.get('container_no_normalized'), ceisa.get('container_no_raw')]
                if ceisa.get('container_nos'): gt_vals.extend(ceisa.get('container_nos'))
                if ceisa.get('containers'): gt_vals.extend([c.get('no') for c in ceisa.get('containers')])
            elif field == 'hs_code':
                gt_vals = [ceisa.get('hs_code'), ceisa.get('hs_code_raw')]
                if ceisa.get('hs_codes_normalized'): gt_vals.extend(ceisa.get('hs_codes_normalized'))
                if ceisa.get('hs_codes_raw'): gt_vals.append(ceisa.get('hs_codes_raw'))
            elif field == 'beratKotor':
                gt_vals = [ceisa.get('beratKotor'), ceisa.get('beratKotor_raw')]
            elif field == 'tglBl':
                gt_vals = [ceisa.get('tglBl'), ceisa.get('tglBl_raw')]
            elif field == 'namaKapal':
                gt_vals = [ceisa.get('namaKapal'), ceisa.get('namaKapal_raw')]
            elif field == 'voyageNumber':
                gt_vals = [ceisa.get('voyageNumber'), ceisa.get('voyageNumber_raw')]
            else:
                gt_vals = [ceisa.get(field)]

            # 2. RESOLVE ALIASES FROM PREDICTION
            pred_val = pred_json.get(field)
            if not pred_val and field == 'pelabuhan_muat': pred_val = pred_json.get('pelabuhanMuat')
            if not pred_val and field == 'pelabuhan_bongkar': pred_val = pred_json.get('pelabuhanBongkar')
            if not pred_val and field == 'container_no': pred_val = pred_json.get('containerNo')
            if not pred_val and field == 'hs_code': pred_val = pred_json.get('hsCode')
            if not pred_val and field == 'nomorBl': pred_val = pred_json.get('booking_no') or pred_json.get('bookingNo')

            # 3. HANDLE NESTED PREDICTION LISTS (Extracting from inside container objects if missing at root)
            containers = pred_json.get('containerNo') or pred_json.get('container_no')
            if not pred_val and isinstance(containers, list):
                if field == 'beratKotor':
                    total = 0
                    for c in containers:
                        if isinstance(c, dict):
                            try: total += float(str(c.get('berat') or c.get('beratKotor')).replace(',', ''))
                            except: pass
                    if total > 0: pred_val = str(total)
                elif field in ['hs_code', 'hsCode']:
                    hss = [str(c.get('hsCode') or c.get('hs_code')) for c in containers if isinstance(c, dict) and (c.get('hsCode') or c.get('hs_code'))]
                    if hss: pred_val = ','.join(hss)
                elif field == 'namaKapal':
                    kpl = [str(c.get('namaKapal')) for c in containers if isinstance(c, dict) and c.get('namaKapal')]
                    if kpl: pred_val = kpl[0]
                elif field == 'voyageNumber':
                    voy = [str(c.get('voyageNumber')) for c in containers if isinstance(c, dict) and c.get('voyageNumber')]
                    if voy: pred_val = voy[0]
                    
            if isinstance(pred_val, list):
                strs = []
                for item in pred_val:
                    if isinstance(item, dict):
                        strs.append(str(item.get('noContainer') or item.get('no') or ''))
                    else:
                        strs.append(str(item))
                pred_val = ','.join([s for s in strs if s])

            score = calculate_anls(gt_vals, pred_val)
            field_scores[field].append(score)
            
            # Output Formatting
            best_gt = [g for g in gt_vals if g is not None]
            best_gt_str = str(best_gt[0]) if best_gt else 'None'
            if len(best_gt) > 1: best_gt_str += f' (or {len(best_gt)-1} others)'
            
            icon = '\u2705' if score >= 0.85 else '\u274c'
            print(f'  {icon} {field:20}: ANLS={score:.3f}  GT={best_gt_str}  Pred={str(pred_val)!r}')

        insw_match = (insw_pred == insw_gt)
        insw_correct.append(insw_match)
        insw_icon = '\u2705' if insw_match else '\u274c'
        print(f'  {insw_icon} insw_flag          : GT={insw_gt} | Pred={insw_pred}\n')

    print('\n' + '='*50)
    print('=== HASIL AKHIR WEIGHTED ANLS SCORE ===')
    print('='*50)

    weighted_scores = []
    for field, weight in EVAL_FIELDS.items():
        scores = field_scores[field]
        if not scores: continue
        avg = np.mean(scores)
        weighted_scores.extend([avg] * int(weight * 2))
        status = '\u2705 LULUS' if avg >= 0.85 else '\u274c GAGAL'
        print(f'{field:22} : {avg:.4f}  {status}')

    if not weighted_scores:
        print('\u26a0\ufe0f  Tidak ada dokumen yang berhasil dievaluasi!')
        return

    insw_accuracy = np.mean(insw_correct) if insw_correct else 0.0
    insw_status   = '\u2705 LULUS' if insw_accuracy >= 0.9 else '\u274c GAGAL'
    print(f'{"INSW Flag Detection":22} : {insw_accuracy:.4f}  {insw_status}')

    final_anls = np.mean(weighted_scores)
    print(f'\nWEIGHTED ANLS RATA-RATA: {final_anls:.4f}')
    print(f'INSW Detection Accuracy: {insw_accuracy:.4f}\n')
    
    if final_anls >= 0.85:
        print('\U0001f389 MODEL LULUS NFR-007 (Akurasi >= 85%)! Siap deployment.')
    else:
        print(f'\u26a0\ufe0f  Akurasi {final_anls:.1%} belum mencapai target 85%.')

evaluate_all()
